In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-5-1-plasma

Plot age-stratified inter-tissue communication and plasma panels from cached results.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


This notebook reads cached CellPhoneDB results after significance filtering. It plots tissue communication for ages <60 and >60 and age-specific ligand–receptor routes. It does not rerun CellPhoneDB.

## 1. Environment, inputs and plot settings

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# Matplotlib uses its platform-default configuration directory.

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle
import numpy as np
import pandas as pd
from IPython.display import Image, display

from figure.figure5 import cellphonedb_interaction_count_network as network


ROOT = Path(input_path("0-figure-code"))
AGE_DIR = (
    ROOT / "0-result-6-2-plasma-data-output" / "22_cellphonedb_tissue_pseudobulk"
    / "24_age_stratified_lt60_gt60"
)
OUTPUT_DIR = Path(output_path("figure5/plasma/figure4g_style"))
FIGURE_STEM = OUTPUT_DIR / "Figure4G_style_lt60_gt60_tissue_interaction"
CHANGED_ROUTE_FILE = OUTPUT_DIR / "Figure4G_age_group_specific_lr_routes.csv"

INK = "#171717"
MUTED = "#596474"
PINKS = ["#FDECEF", "#F7C8D1", "#EC899F", "#D72D62"]
GROUPS = {
    "lt60": {"label": "Age < 60 years", "short": "<60"},
    "gt60": {"label": "Age > 60 years", "short": ">60"},
}


## 2. Load age-stratified tissues, routes and edges

In [ ]:
def _label(tissue: str) -> str:
    return tissue.replace("_", " ").capitalize()


def _load_data():
    tissues = pd.read_csv(AGE_DIR / "24_shared_tissues.csv")["shared_tissue"].astype(str).tolist()
    summary = pd.read_csv(AGE_DIR / "24_age_group_input_summary.csv")
    results = {}
    for group in GROUPS:
        edges = pd.read_csv(AGE_DIR / group / f"24_{group}_tissue_edges.csv")
        routes = pd.read_csv(AGE_DIR / group / f"24_{group}_significant_routes.csv")
        results[group] = (routes, edges)
    return tissues, summary, results


def _circular_positions(tissues: list[str]) -> dict[str, np.ndarray]:
    """Shared clockwise positions with liver fixed at three o'clock."""
    ordered = ["liver"] + sorted(tissue for tissue in tissues if tissue != "liver")
    angles = np.linspace(0, -2 * np.pi, len(ordered), endpoint=False)
    return {
        tissue: np.array([np.cos(angle), np.sin(angle)])
        for tissue, angle in zip(ordered, angles)
    }


def _edge_width(count: float) -> float:
    if count <= 0:
        return 0.45
    return 0.70 + 0.90 * float(count)


def _draw_self_loop(ax, xy, color, width):
    radial = xy / max(np.linalg.norm(xy), 1e-9)
    tangent = np.array([-radial[1], radial[0]])
    start = xy + radial * 0.035 + tangent * 0.072
    end = xy + radial * 0.035 - tangent * 0.072
    ax.add_patch(FancyArrowPatch(
        start, end,
        connectionstyle="arc3,rad=-2.15",
        arrowstyle="-|>", mutation_scale=8.0,
        linewidth=width, color=color, alpha=0.93,
        shrinkA=1, shrinkB=1, zorder=2,
    ))


def _node_area(flow: float, max_flow: float) -> float:
    """Area in points squared; shared across panels for direct comparison."""
    if max_flow <= 0:
        return 78.0
    return 78.0 + 305.0 * np.sqrt(max(float(flow), 0.0) / max_flow)


def _age_group_specific_routes(results: dict) -> dict[str, pd.DataFrame]:
    """Return routes significant in one age group but absent from the other."""
    route_columns = [
        "source_tissue", "target_tissue", "ligand_gene",
        "ligand_receptor_pair",
    ]
    route_frames = {
        group: routes[route_columns].drop_duplicates().copy()
        for group, (routes, _) in results.items()
    }
    route_sets = {
        group: set(map(tuple, frame.astype(str).to_numpy()))
        for group, frame in route_frames.items()
    }
    changed: dict[str, pd.DataFrame] = {}
    for group, other in (("lt60", "gt60"), ("gt60", "lt60")):
        unique_keys = route_sets[group] - route_sets[other]
        frame = pd.DataFrame(sorted(unique_keys), columns=route_columns)
        frame["age_group_specific"] = GROUPS[group]["label"]
        changed[group] = frame
    return changed


In [ ]:
tissues, input_summary, age_results = _load_data()
print(f'Shared tissues: {len(tissues)}')
display(input_summary)
for age_group, (routes, edges) in age_results.items():
    print(f'{GROUPS[age_group]["label"]}: {len(routes)} routes, {len(edges)} directed edges')


## 3. Age-stratified tissue communication network

In [ ]:
def _draw_changed_route_labels(ax, routes, positions):
    """Place compact labels beside age-group-specific tissue routes."""
    if routes.empty:
        return
    for index, row in enumerate(
        routes.sort_values(["target_tissue", "ligand_receptor_pair"])
        .itertuples(index=False)
    ):
        start = positions[str(row.source_tissue)]
        end = positions[str(row.target_tissue)]
        vector = end - start
        norm_vector = max(float(np.linalg.norm(vector)), 1e-9)
        perpendicular = np.array([-vector[1], vector[0]]) / norm_vector
        # Alternating, restrained offsets keep nearby radial labels separate.
        side = 1.0 if index % 2 == 0 else -1.0
        anchor = 0.43 * start + 0.57 * end + side * 0.060 * perpendicular
        pair_label = str(row.ligand_receptor_pair).replace(" → ", "→")
        ax.text(
            anchor[0], anchor[1], pair_label,
            ha="center", va="center", fontsize=5.55,
            fontweight="semibold", color="#8F3952", zorder=8,
            bbox={
                "boxstyle": "round,pad=0.18", "facecolor": "white",
                "edgecolor": "#E8A7B7", "linewidth": 0.45, "alpha": 0.94,
            },
        )


def _draw_panel(
    ax, group, positions, tissues, edges, flows, max_flow, cmap, norm,
    summary, changed_routes,
):
    ax.set_aspect("equal")
    ax.set_xlim(-1.48, 1.48)
    ax.set_ylim(-1.43, 1.43)
    ax.axis("off")

    # Chords first, then nodes. The two panels use identical coordinates.
    for row in edges.sort_values(["lr_pair_count", "target_tissue"]).itertuples(index=False):
        start = positions[str(row.source_tissue)]
        end = positions[str(row.target_tissue)]
        count = float(row.lr_pair_count)
        color = cmap(norm(count))
        width = _edge_width(count)
        if row.source_tissue == row.target_tissue:
            _draw_self_loop(ax, start, color, width)
        else:
            ax.add_patch(FancyArrowPatch(
                start, end,
                connectionstyle="arc3,rad=0",
                arrowstyle="-|>", mutation_scale=7.2,
                linewidth=width, color=color, alpha=0.91,
                shrinkA=8.0, shrinkB=8.0, zorder=2,
            ))

    for tissue in tissues:
        xy = positions[tissue]
        size = _node_area(flows.get(tissue, 0.0), max_flow)
        ax.scatter(*xy, s=size + 42, facecolor="white", edgecolor="white", zorder=4)
        ax.scatter(
            *xy, s=size, facecolor=network.TISSUE_COLORS[tissue],
            edgecolor="#333333", linewidth=0.65, zorder=5,
        )

        # Direct tissue labels replace the detached color legend. Labels are
        # pushed radially outside the node circle and aligned away from it.
        label_xy = xy * 1.19
        if abs(float(xy[0])) < 0.18:
            horizontal_alignment = "center"
        else:
            horizontal_alignment = "left" if xy[0] > 0 else "right"
        if abs(float(xy[1])) < 0.18:
            vertical_alignment = "center"
        else:
            vertical_alignment = "bottom" if xy[1] > 0 else "top"
        ax.text(
            label_xy[0], label_xy[1], _label(tissue),
            ha=horizontal_alignment, va=vertical_alignment,
            fontsize=7.0, color=INK, zorder=6, clip_on=False,
        )

    _draw_changed_route_labels(ax, changed_routes, positions)

    row = summary.loc[summary["age_group"].eq(group)].iloc[0]
    ax.text(
        0.5, 1.055, GROUPS[group]["label"], transform=ax.transAxes,
        ha="center", va="bottom", fontsize=12.6, color=INK,
    )
    ax.text(
        0.5, 1.010,
        f"{int(row.n_unique_donors_shared_tissues)} donors  |  "
        f"{int(len(edges))} directed edges  |  "
        f"{int(edges.lr_pair_count.sum())} interaction pairs  |  "
        f"{int(len(changed_routes))} age-specific pairs",
        transform=ax.transAxes, ha="center", va="bottom",
        fontsize=7.1, color=MUTED,
    )


## 4. Network legends and layout

In [ ]:
def _draw_tissue_legend(fig, tissues):
    legend = fig.add_axes([0.095, 0.075, 0.650, 0.115])
    legend.set_xlim(0, 1)
    legend.set_ylim(0, 1)
    legend.axis("off")
    ncols = 7
    for index, tissue in enumerate(tissues):
        row, col = divmod(index, ncols)
        x = 0.015 + col * 0.141
        y = 0.70 - row * 0.43
        legend.scatter(x, y, s=66, facecolor=network.TISSUE_COLORS[tissue],
                       edgecolor="white", linewidth=0.5)
        legend.text(x + 0.022, y, _label(tissue), va="center", ha="left",
                    fontsize=8.2, color=INK)


def _draw_count_legends(fig, cmap, norm, max_count, max_flow):
    side = fig.add_axes([0.805, 0.205, 0.165, 0.565])
    side.set_xlim(0, 1)
    side.set_ylim(0, 1)
    side.axis("off")
    side.text(0.0, 0.99, "Interaction counts", fontsize=9.7,
              fontweight="bold", color=INK, va="top")

    cax = fig.add_axes([0.827, 0.500, 0.018, 0.220])
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, cax=cax, orientation="vertical")
    cbar.set_ticks(np.arange(0, int(max_count) + 1))
    cbar.ax.tick_params(labelsize=7.6, width=0.6, length=2.2)
    cbar.set_label("Interaction counts", fontsize=8.1, labelpad=5)
    cbar.outline.set_visible(False)

    side.text(0.0, 0.50, "Interaction counts", fontsize=9.7,
              fontweight="bold", color=INK, va="top")
    for index, count in enumerate(range(int(max_count), -1, -1)):
        y = 0.41 - index * 0.074
        side.plot([0.05, 0.34], [y, y], color="#111111",
                  linewidth=_edge_width(count), solid_capstyle="butt")
        side.text(0.46, y, str(count), va="center", ha="left",
                  fontsize=8.2, color=INK)

    node = fig.add_axes([0.805, 0.055, 0.165, 0.145])
    node.set_xlim(0, 1)
    node.set_ylim(0, 1)
    node.axis("off")
    node.text(0.0, 0.98, "Node area", fontsize=9.7,
              fontweight="bold", color=INK, va="top")
    node.text(0.0, 0.76, "Total significant LR-pair flow",
              fontsize=7.2, color=MUTED, va="top")
    examples = [0.0, float(max_flow)]
    for x, value in zip([0.22, 0.67], examples):
        node.scatter(x, 0.34, s=_node_area(value, max_flow),
                     facecolor="#EAF0F4", edgecolor="#303A44",
                     linewidth=0.65, clip_on=False)
        node.text(x, 0.03, f"{value:g}", ha="center", va="bottom",
                  fontsize=7.8, color=INK)


In [ ]:
def draw_figure():
    tissues, summary, results = _load_data()
    positions = _circular_positions(tissues)
    max_count = max(
        1.0,
        max(float(edges["lr_pair_count"].max()) for _, edges in results.values()),
    )
    flows = {
        group: network.flow_values(edges)
        for group, (_, edges) in results.items()
    }
    max_flow = max(
        [value for group_flows in flows.values() for value in group_flows.values()]
        or [1.0]
    )
    cmap = LinearSegmentedColormap.from_list("figure4g_pink", PINKS)
    norm = Normalize(vmin=0, vmax=max_count)
    changed_routes = _age_group_specific_routes(results)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    pd.concat(changed_routes.values(), ignore_index=True).to_csv(
        CHANGED_ROUTE_FILE, index=False, encoding="utf-8-sig"
    )

    mpl.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 9,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    })
    fig = plt.figure(figsize=(12.4, 5.65), facecolor="white")
    axes = {
        "lt60": fig.add_axes([0.050, 0.120, 0.330, 0.730]),
        "gt60": fig.add_axes([0.415, 0.120, 0.330, 0.730]),
    }

    for group in ["lt60", "gt60"]:
        routes, edges = results[group]
        _draw_panel(
            axes[group], group, positions, tissues, edges,
            flows[group], max_flow, cmap, norm, summary, changed_routes[group],
        )

    _draw_count_legends(fig, cmap, norm, max_count, max_flow)
    fig.text(
        0.070, 0.022,
        "CellPhoneDB permutation P < 0.05 and BH-FDR < 0.05; identical tissue positions and interaction-count scales across age groups. "
        "Arrows indicate SAGE-context source tissue → statistically supported receiver tissue. "
        "Boxed labels denote age-group-specific LR pairs; routes shared by both groups are not labelled.",
        ha="left", va="bottom", fontsize=6.8, color=MUTED,
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for extension in ["png", "pdf", "svg"]:
        fig.savefig(
            FIGURE_STEM.with_suffix(f".{extension}"),
            dpi=600 if extension == "png" else None,
            bbox_inches="tight", facecolor="white",
        )
    return fig, FIGURE_STEM


def main():
    figure, stem = draw_figure()
    plt.close(figure)
    print(f"Saved Figure 4G-style network: {stem}.png/.pdf/.svg")
    return {"figure_stem": stem}


if __name__ == "__main__":
    main()


## 5. Generate and save the communication network

In [ ]:
from IPython.display import Image, display

figure4g_results = main()
figure_png = figure4g_results['figure_stem'].with_suffix('.png')
figure_pdf = figure4g_results['figure_stem'].with_suffix('.pdf')
figure_svg = figure4g_results['figure_stem'].with_suffix('.svg')

print('PNG:', figure_png)
print('PDF:', figure_pdf)
print('SVG:', figure_svg)
display(Image(filename=str(figure_png), width=1500))


In [ ]:
from matplotlib.colors import TwoSlopeNorm

FIGURE4HI_OUTPUT_DIR = Path(output_path("figure5/plasma/figure4hi_age_comparison"))
FIGURE_I_STEM = FIGURE4HI_OUTPUT_DIR / 'Figure4I_lt60_gt60_lr_pair_count_difference'
COMPARISON_FILE = FIGURE4HI_OUTPUT_DIR / 'Figure4I_lt60_gt60_lr_pair_count_comparison.csv'
PAIR_PRESENCE_FILE = FIGURE4HI_OUTPUT_DIR / 'Figure4I_lt60_gt60_lr_pair_presence_change.csv'
EXCLUSIVE_CHANGE_FILE = FIGURE4HI_OUTPUT_DIR / 'Figure4I_lt60_gt60_exclusive_lr_pair_changes.csv'
FIGURE4HI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _publication_defaults() -> None:
    mpl.rcParams.update(
        {
            "font.family": "DejaVu Sans",
            "font.size": 8.5,
            "axes.edgecolor": "#26313C",
            "axes.labelcolor": "#26313C",
            "xtick.color": "#26313C",
            "ytick.color": "#26313C",
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )


def _save(fig: plt.Figure, stem: Path) -> None:
    stem.parent.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "pdf", "svg"):
        fig.savefig(
            stem.with_suffix(f".{extension}"),
            dpi=600 if extension == "png" else None,
            bbox_inches="tight",
            facecolor="white",
        )
    plt.close(fig)

def _receiver_order(significant: dict[str, pd.DataFrame]) -> list[str]:
    observed = set().union(
        *(set(frame["target_tissue"].dropna().astype(str)) for frame in significant.values())
    )
    ordered = [t for t in network.FIG4H_RAINBOW_TISSUE_ORDER if t in observed]
    ordered.extend(sorted(observed - set(ordered)))
    return ordered


In [ ]:
def build_lr_pair_count_comparison(
    significant: dict[str, pd.DataFrame]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Compare distinct significant LR-pair counts, never expression means."""
    route_keys = sorted({
        (row.source_tissue, row.target_tissue)
        for frame in significant.values()
        for row in frame[["source_tissue", "target_tissue"]].itertuples(index=False)
    })
    pair_universe = sorted({
        pair
        for frame in significant.values()
        for pair in frame["ligand_receptor_pair"].dropna().astype(str).unique()
    })
    pair_sets = {
        group: {
            (row.source_tissue, row.target_tissue, row.ligand_receptor_pair)
            for row in frame[["source_tissue", "target_tissue", "ligand_receptor_pair"]]
            .drop_duplicates().itertuples(index=False)
        }
        for group, frame in significant.items()
    }

    pair_rows = []
    route_rows = []
    for source_tissue, target_tissue in route_keys:
        lt_pairs = {
            pair for pair in pair_universe
            if (source_tissue, target_tissue, pair) in pair_sets["lt60"]
        }
        gt_pairs = {
            pair for pair in pair_universe
            if (source_tissue, target_tissue, pair) in pair_sets["gt60"]
        }
        for pair in pair_universe:
            present_lt60 = pair in lt_pairs
            present_gt60 = pair in gt_pairs
            pair_rows.append({
                "source_tissue": source_tissue,
                "target_tissue": target_tissue,
                "ligand_receptor_pair": pair,
                "significant_lt60": present_lt60,
                "significant_gt60": present_gt60,
                "presence_delta_gt60_minus_lt60": int(present_gt60) - int(present_lt60),
                "present_in_either_group": present_lt60 or present_gt60,
            })
        shared = sorted(lt_pairs & gt_pairs)
        lost = sorted(lt_pairs - gt_pairs)
        gained = sorted(gt_pairs - lt_pairs)
        route_rows.append({
            "source_tissue": source_tissue,
            "target_tissue": target_tissue,
            "n_distinct_lr_pairs_lt60": len(lt_pairs),
            "n_distinct_lr_pairs_gt60": len(gt_pairs),
            "delta_lr_pair_count_gt60_minus_lt60": len(gt_pairs) - len(lt_pairs),
            "shared_lr_pairs": "; ".join(shared),
            "lost_at_gt60": "; ".join(lost),
            "gained_at_gt60": "; ".join(gained),
        })
    pair_presence = pd.DataFrame(pair_rows)
    route_counts = pd.DataFrame(route_rows)
    pair_presence.to_csv(PAIR_PRESENCE_FILE, index=False, encoding="utf-8-sig")
    route_counts.to_csv(COMPARISON_FILE, index=False, encoding="utf-8-sig")
    pair_presence.loc[
        pair_presence["presence_delta_gt60_minus_lt60"].ne(0)
    ].to_csv(EXCLUSIVE_CHANGE_FILE, index=False, encoding="utf-8-sig")
    return pair_presence, route_counts


In [ ]:
significant_by_age = {
    group: routes.copy()
    for group, (routes, edges) in age_results.items()
}
receiver_order = _receiver_order(significant_by_age)
pair_presence, route_count_comparison = build_lr_pair_count_comparison(significant_by_age)

age_specific_pairs = pair_presence.loc[
    pair_presence['presence_delta_gt60_minus_lt60'].ne(0)
].copy()
display(age_specific_pairs[[
    'source_tissue', 'target_tissue', 'ligand_receptor_pair',
    'significant_lt60', 'significant_gt60',
    'presence_delta_gt60_minus_lt60',
]])


In [ ]:
def plot_figure4i(
    pair_presence: pd.DataFrame,
    route_counts: pd.DataFrame,
    receiver_order: list[str],
) -> None:
    """Cell-style compact matrix containing age-group-specific routes only."""
    _publication_defaults()
    changed_pairs = pair_presence.loc[
        pair_presence["presence_delta_gt60_minus_lt60"].ne(0)
    ].copy()
    if changed_pairs.empty:
        raise ValueError("No age-group-specific ligand–receptor routes were detected.")

    observed_tissues = set(changed_pairs["target_tissue"])
    tissues = [t for t in receiver_order if t in observed_tissues]
    tissues.extend(sorted(observed_tissues - set(tissues)))
    pair_order_preference = [
        "VTN → ITGAV+ITGB1",
        "VTN → ITGAV+ITGB3",
        "VTN → ITGA2B+ITGB3",
    ]
    observed_pairs = set(changed_pairs["ligand_receptor_pair"])
    pairs = [pair for pair in pair_order_preference if pair in observed_pairs]
    pairs.extend(sorted(observed_pairs - set(pairs)))
    changed_pairs["target_tissue"] = pd.Categorical(
        changed_pairs["target_tissue"], tissues, ordered=True
    )
    changed_pairs["ligand_receptor_pair"] = pd.Categorical(
        changed_pairs["ligand_receptor_pair"], pairs, ordered=True
    )
    changed_pairs = changed_pairs.sort_values(["ligand_receptor_pair", "target_tissue"])
    xmap = {tissue: index for index, tissue in enumerate(tissues)}
    ymap = {pair: index for index, pair in enumerate(pairs)}

    norm = TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
    cmap = LinearSegmentedColormap.from_list(
        "lr_count_delta", ["#4C78B8", "#FAF7F7", "#D81B60"]
    )

    # A deliberately compact aspect ratio and sparse annotation system mirror
    # the Cell reference panel. All observed changes have magnitude one, so dot
    # area is fixed and carries no redundant quantitative encoding.
    fig = plt.figure(figsize=(5.95, 4.25), facecolor="white")
    ax = fig.add_axes([0.08, 0.255, 0.58, 0.46])
    x = changed_pairs["target_tissue"].map(xmap).to_numpy(dtype=float)
    y = changed_pairs["ligand_receptor_pair"].map(ymap).to_numpy(dtype=float)
    delta = changed_pairs["presence_delta_gt60_minus_lt60"].to_numpy(dtype=float)
    ax.scatter(
        x, y, s=148, marker="o", c=delta, cmap=cmap, norm=norm,
        edgecolor="#20242A", linewidth=0.75, zorder=3,
    )

    ax.set_xlim(-0.55, len(tissues) - 0.45)
    ax.set_ylim(len(pairs) - 0.45, -0.55)
    ax.set_xticks(range(len(tissues)))
    ax.set_xticklabels(
        [f"Liver : {t.replace('_', ' ').capitalize()}" for t in tissues],
        rotation=90, ha="left", va="bottom", fontsize=7.5,
    )
    ax.xaxis.tick_top()
    ax.tick_params(axis="x", length=4.0, width=0.75, pad=5)
    ax.set_yticks(range(len(pairs)))
    ax.set_yticklabels(
        [r"$\bf{" + p.split(" → ")[0] + "}$ : " + p.split(" → ")[1] for p in pairs],
        fontsize=8.7,
    )
    ax.yaxis.tick_right()
    ax.tick_params(axis="y", length=4.0, width=0.75, pad=8)
    for spine in ax.spines.values():
        spine.set_linewidth(0.85)
        spine.set_color("#20242A")
    ax.grid(False)

    colorbar_ax = fig.add_axes([0.18, 0.095, 0.30, 0.030])
    scalar_map = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    scalar_map.set_array([])
    colorbar = fig.colorbar(scalar_map, cax=colorbar_ax, orientation="horizontal")
    colorbar.outline.set_visible(False)
    colorbar.ax.tick_params(labelsize=7.8, length=2.8, width=0.65)
    colorbar.set_ticks([-1, 0, 1])
    colorbar.ax.set_xticklabels(["−1", "0", "+1"])
    fig.text(
        0.18, 0.145, "Δ interaction count (>60 − <60)",
        fontsize=8.7, color="#111111",
    )

    n_lost = int((delta < 0).sum())
    n_gained = int((delta > 0).sum())
    fig.text(0.018, 0.965, "I", fontsize=19.0, fontweight="bold", color="#0D1117")
    fig.text(
        0.70, 0.115,
        f"Blue: <60-specific ({n_lost})  |  Magenta: >60-specific ({n_gained})\n"
        "Shared significant routes are omitted",
        fontsize=6.8, color=MUTED, ha="left", va="center", linespacing=1.35,
    )
    _save(fig, FIGURE_I_STEM)


In [ ]:
plot_figure4i(pair_presence, route_count_comparison, receiver_order)

figure_i_png = FIGURE_I_STEM.with_suffix('.png')
figure_i_pdf = FIGURE_I_STEM.with_suffix('.pdf')
figure_i_svg = FIGURE_I_STEM.with_suffix('.svg')
print('PNG:', figure_i_png)
print('PDF:', figure_i_pdf)
print('SVG:', figure_i_svg)
display(Image(filename=str(figure_i_png), width=900))


In [ ]:
from scipy import stats

LIVER_VTN_CELL_FILE = (
    ROOT / "0-result-6-2-plasma-data-output"
    / "11_source_ligand_single_cell_expression.csv.gz"
)
EXPRESSION_PANEL_DIR = Path(output_path("figure5/plasma/figure4hi_age_comparison"))
LIVER_VTN_PANEL_STEM = EXPRESSION_PANEL_DIR / "Figure_panel_e_liver_VTN_expression"
EXPRESSION_PANEL_DIR.mkdir(parents=True, exist_ok=True)


def plot_liver_vtn_single_cell_panel():
    data = pd.read_csv(LIVER_VTN_CELL_FILE, encoding="utf-8-sig")
    data = data.loc[
        data["tissue_key"].eq("liver") & data["gene"].eq("VTN")
    ].copy()
    data["age"] = pd.to_numeric(data["age"], errors="coerce")
    data["expression"] = pd.to_numeric(data["expression"], errors="coerce")
    data = data.dropna(subset=["age", "expression", "donor"])

    ages = sorted(data["age"].astype(int).unique())
    if len(ages) != 2:
        raise ValueError(f"Expected two liver donor ages, found {ages}")
    younger_age, older_age = ages
    younger = data.loc[data["age"].eq(younger_age), "expression"].to_numpy(float)
    older = data.loc[data["age"].eq(older_age), "expression"].to_numpy(float)

    test = stats.mannwhitneyu(
        older, younger, alternative="two-sided", method="asymptotic"
    )
    cliffs_delta = 2 * test.statistic / (len(older) * len(younger)) - 1
    p_text = "P < 2.2e-16" if test.pvalue < 2.2e-16 else f"P = {test.pvalue:.3g}"

    with plt.rc_context({
        "font.family": "DejaVu Sans", "font.size": 8.3,
        "axes.linewidth": 0.85, "pdf.fonttype": 42,
        "ps.fonttype": 42, "svg.fonttype": "none",
    }):
        fig, ax = plt.subplots(figsize=(4.25, 3.15), facecolor="white")
        box = ax.boxplot(
            [younger, older], positions=[0, 1], widths=0.50,
            patch_artist=True, showfliers=False, whis=(5, 95),
            medianprops={"color": "#263238", "linewidth": 1.35},
            whiskerprops={"color": "#526170", "linewidth": 0.9},
            capprops={"color": "#526170", "linewidth": 0.9},
            boxprops={"linewidth": 0.95},
        )
        for patch, face, edge in zip(
            box["boxes"], ["#B9D8E8", "#2187B8"], ["#4D7891", "#075A87"]
        ):
            patch.set_facecolor(face)
            patch.set_edgecolor(edge)
            patch.set_alpha(0.96)

        y_max = max(float(np.max(younger)), float(np.max(older)))
        bracket_y, bracket_h = y_max + 0.30, 0.18
        ax.plot(
            [0, 0, 1, 1],
            [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y],
            color="#22272B", linewidth=0.85, clip_on=False,
        )
        ax.text(0.5, bracket_y + bracket_h + 0.08, p_text,
                ha="center", va="bottom", fontsize=8.0)

        ax.set_xticks([0, 1], [f"{younger_age} years", f"{older_age} years"])
        ax.set_ylabel("VTN expression in liver (log-normalized)")
        ax.set_title("Single-cell transcriptomics", fontsize=10.3, pad=11)
        ax.set_ylim(0, bracket_y + bracket_h + 0.52)
        ax.grid(False)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(axis="x", length=0, pad=6)
        fig.subplots_adjust(left=0.18, right=0.98, bottom=0.16, top=0.83)

        for suffix in ("png", "pdf", "svg"):
            fig.savefig(
                LIVER_VTN_PANEL_STEM.with_suffix(f".{suffix}"),
                dpi=600 if suffix == "png" else None,
                bbox_inches="tight", facecolor="white",
            )

    summary = pd.DataFrame({
        "age": [younger_age, older_age],
        "n_cells": [len(younger), len(older)],
        "mean_expression": [younger.mean(), older.mean()],
        "median_expression": [np.median(younger), np.median(older)],
        "mannwhitney_p": [test.pvalue, test.pvalue],
        "cliffs_delta_older_vs_younger": [cliffs_delta, cliffs_delta],
    })
    summary.to_csv(
        LIVER_VTN_PANEL_STEM.with_name(LIVER_VTN_PANEL_STEM.name + "_data.csv"),
        index=False,
    )
    return fig, summary


In [ ]:
liver_vtn_figure, liver_vtn_summary = plot_liver_vtn_single_cell_panel()
display(liver_vtn_summary)
display(Image(filename=str(LIVER_VTN_PANEL_STEM.with_suffix('.png')), width=650))
plt.close(liver_vtn_figure)


In [ ]:
PLASMA_VTN_VALUE_FILE = (
    ROOT / "0-result-7-2-gene-expression-age-output"
    / "VTN_VEGFA_plasma_age_values.csv"
)
PLASMA_VTN_PANEL_STEM = EXPRESSION_PANEL_DIR / "Figure_panel_f_plasma_VTN_abundance"


def plot_plasma_vtn_young_old_panel():
    data = pd.read_csv(PLASMA_VTN_VALUE_FILE)
    data = data.loc[data["gene"].eq("VTN")].copy()
    data["age"] = pd.to_numeric(data["age"], errors="coerce")
    data["abundance"] = pd.to_numeric(data["abundance"], errors="coerce")
    data = data.dropna(subset=["age", "abundance"])

    young = data.loc[data["age"].le(30), "abundance"].to_numpy(float)
    old = data.loc[data["age"].ge(46), "abundance"].to_numpy(float)
    test = stats.ttest_ind(old, young, equal_var=False, alternative="two-sided")
    mann_whitney = stats.mannwhitneyu(old, young, alternative="two-sided")
    cliffs_delta = 2 * mann_whitney.statistic / (len(old) * len(young)) - 1

    with plt.rc_context({
        "font.family": "DejaVu Sans", "font.size": 8.3,
        "axes.linewidth": 0.85, "pdf.fonttype": 42,
        "ps.fonttype": 42, "svg.fonttype": "none",
    }):
        fig, ax = plt.subplots(figsize=(4.25, 3.15), facecolor="white")
        box = ax.boxplot(
            [young, old], positions=[0, 1], widths=0.50,
            patch_artist=True, showfliers=False, whis=(5, 95),
            medianprops={"color": "#263238", "linewidth": 1.35},
            whiskerprops={"color": "#526170", "linewidth": 0.9},
            capprops={"color": "#526170", "linewidth": 0.9},
            boxprops={"linewidth": 0.95},
        )
        for patch, face, edge in zip(
            box["boxes"], ["#F3C5D0", "#DF8299"], ["#C87489", "#B6506C"]
        ):
            patch.set_facecolor(face)
            patch.set_edgecolor(edge)
            patch.set_alpha(0.97)

        rng = np.random.default_rng(20250812)
        for position, values in enumerate([young, old]):
            jitter = np.clip(rng.normal(0, 0.075, len(values)), -0.18, 0.18)
            ax.scatter(
                position + jitter, values, s=9.0,
                color="#9DA5AA", edgecolor="none", alpha=0.45, zorder=3,
            )

        all_values = np.concatenate([young, old])
        y_min, y_max = float(all_values.min()), float(all_values.max())
        span = y_max - y_min
        bracket_y, bracket_h = y_max + 0.07 * span, 0.03 * span
        ax.plot(
            [0, 0, 1, 1],
            [bracket_y, bracket_y + bracket_h, bracket_y + bracket_h, bracket_y],
            color="#22272B", linewidth=0.85, clip_on=False,
        )
        ax.text(0.5, bracket_y + bracket_h + 0.018 * span,
                f"P = {test.pvalue:.3f}", ha="center", va="bottom", fontsize=8.0)

        ax.set_xticks([0, 1], ["Young", "Old"])
        ax.set_ylabel("Plasma VTN protein abundance")
        ax.set_title("Plasma proteomics", fontsize=10.3, pad=11)
        ax.set_ylim(y_min - 0.07 * span, y_max + 0.20 * span)
        ax.grid(False)
        ax.spines[["top", "right"]].set_visible(False)
        ax.tick_params(axis="x", length=0, pad=6)
        fig.subplots_adjust(left=0.18, right=0.98, bottom=0.16, top=0.83)

        for suffix in ("png", "pdf", "svg"):
            fig.savefig(
                PLASMA_VTN_PANEL_STEM.with_suffix(f".{suffix}"),
                dpi=600 if suffix == "png" else None,
                bbox_inches="tight", facecolor="white",
            )

    summary = pd.DataFrame({
        "group": ["Young (≤30)", "Old (≥46)"],
        "n": [len(young), len(old)],
        "mean_abundance": [young.mean(), old.mean()],
        "median_abundance": [np.median(young), np.median(old)],
        "welch_t": [test.statistic, test.statistic],
        "welch_p": [test.pvalue, test.pvalue],
        "cliffs_delta_old_vs_young": [cliffs_delta, cliffs_delta],
    })
    summary.to_csv(
        PLASMA_VTN_PANEL_STEM.with_name(PLASMA_VTN_PANEL_STEM.name + "_data.csv"),
        index=False,
    )
    return fig, summary


In [ ]:
plasma_vtn_figure, plasma_vtn_summary = plot_plasma_vtn_young_old_panel()
display(plasma_vtn_summary)
display(Image(filename=str(PLASMA_VTN_PANEL_STEM.with_suffix('.png')), width=650))
plt.close(plasma_vtn_figure)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display


# Input: original single-cell receptor-complex values and selected high-expression cell types.
VTN_RECEPTOR_SOURCE_DIR = Path(
    input_path("2-8.3-shanda/1-feature/1-figure/0-result-6/VTN_stable_high_receptor_celltypes")
)
VTN_RECEPTOR_CELL_FILE = (
    VTN_RECEPTOR_SOURCE_DIR / "Exact_age_stable_VTN_receptor_cell_values.csv.gz"
)
VTN_RECEPTOR_RANK_FILE = (
    VTN_RECEPTOR_SOURCE_DIR / "Stable_high_receptor_celltypes_top5_overall_expression.csv"
)

PANEL_G_OUTPUT_DIR = Path(output_path("figure5/plasma/figure4hi_age_comparison"))
PANEL_G_OUTPUT_STEM = PANEL_G_OUTPUT_DIR / "Figure_panel_g_VTN_receiver_cell_age_remodeling"

DISPLAY_TISSUE_ORDER_G = [
    "blood", "vasculature", "salivary_gland", "spleen", "thymus", "lung"
]
TISSUE_LABEL_G = {
    "blood": "Blood", "vasculature": "Vasculature",
    "salivary_gland": "Salivary gland", "spleen": "Spleen",
    "thymus": "Thymus", "lung": "Lung",
}
PAIR_COLOR_G = {
    "ITGAV+ITGB1": "#2878B5",
    "ITGAV+ITGB3": "#D97732",
    "ITGA2B+ITGB3": "#159B76",
}


def prepare_panel_g_data(top_n_per_tissue=3):
    """Average expression by donor, then summarize tissue and age-group changes."""
    cells = pd.read_csv(VTN_RECEPTOR_CELL_FILE)
    ranks = pd.read_csv(VTN_RECEPTOR_RANK_FILE)

    cells = cells.loc[cells["age"].ne(60)].copy()
    cells["age_group"] = np.where(cells["age"].lt(60), "Age <60", "Age >60")

    selected = (
        ranks.loc[
            ranks["tissue"].isin(DISPLAY_TISSUE_ORDER_G)
            & ranks["rank_within_receiver_tissue"].le(top_n_per_tissue),
            ["tissue", "receptor_complex", "fine_cell_type",
             "rank_within_receiver_tissue", "median_complex_expression"],
        ]
        .drop_duplicates()
    )

    plot_cells = cells.merge(
        selected.drop(columns="median_complex_expression"),
        on=["tissue", "receptor_complex", "fine_cell_type"],
        how="inner", validate="many_to_one",
    )


    # Each plotted observation is a donor mean within tissue and cell type.
    donor_means = (
        plot_cells.groupby(
            ["tissue", "receptor_complex", "fine_cell_type",
             "rank_within_receiver_tissue", "donor", "age", "age_group"],
            observed=True,
        )["complete_receptor_expression"]
        .mean().rename("donor_mean_expression").reset_index()
    )

    group_summary = (
        donor_means.groupby(
            ["tissue", "receptor_complex", "fine_cell_type",
             "rank_within_receiver_tissue", "age_group"],
            observed=True,
        )["donor_mean_expression"]
        .median().rename("group_median").reset_index()
    )
    age = group_summary.pivot_table(
        index=["tissue", "receptor_complex", "fine_cell_type",
               "rank_within_receiver_tissue"],
        columns="age_group", values="group_median", aggfunc="first",
    ).reset_index()
    age.columns.name = None

    table = selected.merge(
        age,
        on=["tissue", "receptor_complex", "fine_cell_type",
            "rank_within_receiver_tissue"],
        how="left", validate="one_to_one",
    )


    # Normalize the upper row within each tissue; absolute levels are not comparable across tissues.
    tissue_max = table.groupby("tissue")["median_complex_expression"].transform("max")
    table["relative_expression_within_tissue"] = np.divide(
        table["median_complex_expression"], tissue_max,
        out=np.zeros(len(table), dtype=float), where=tissue_max.gt(0),
    )


    # The lower row is a normalized difference of donor-median age groups in [-1, 1].
    denominator = table["Age >60"].abs() + table["Age <60"].abs()
    table["normalized_age_difference"] = np.divide(
        table["Age >60"] - table["Age <60"], denominator,
        out=np.zeros(len(table), dtype=float), where=denominator.gt(0),
    )

    table["display_tissue_order"] = table["tissue"].map(
        {tissue: i for i, tissue in enumerate(DISPLAY_TISSUE_ORDER_G)}
    )
    table = table.sort_values(
        ["display_tissue_order", "rank_within_receiver_tissue"], kind="stable"
    ).reset_index(drop=True)
    return table, donor_means


In [ ]:
def _short_panel_g_cell_type(value):
    replacements = {
        "smooth muscle cell": "smooth muscle",
        "pericyte cell": "pericyte",
        "endothelial cell of lymphatic vessel": "lymphatic endothelial",
        "vascular associated smooth muscle cell": "vascular smooth muscle",
        "endothelial cell of artery": "arterial endothelial",
        "capillary endothelial cell": "capillary endothelial",
        "bronchial smooth muscle cell": "bronchial smooth muscle",
        "vein endothelial cell": "venous endothelial",
        "type i pneumocyte": "type I pneumocyte",
    }
    return replacements.get(str(value), str(value))


def plot_panel_g_vtn_receiver_remodeling(table):
    """Draw panel G: receiver-cell localization above and age-related remodeling below."""
    ink, grid = "#17263C", "#E1E7EB"
    down, up, zero = "#3B6FA8", "#C64F73", "#9AA5AE"
    block_colors = ["#F7F9FA", "#FBFCFD"]

    mpl.rcParams.update({
        "font.family": "DejaVu Sans", "font.size": 8,
        "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
    })

    positions, blocks, cursor = {}, {}, 0.0
    for tissue in DISPLAY_TISSUE_ORDER_G:
        one = table.loc[table["tissue"].eq(tissue)].sort_values(
            "rank_within_receiver_tissue"
        )
        start = cursor
        for row in one.itertuples(index=False):
            positions[(tissue, row.fine_cell_type)] = cursor
            cursor += 1
        blocks[tissue] = (start, cursor - 1)
        cursor += 1.20

    fig = plt.figure(figsize=(12.0, 5.45), facecolor="white")
    gs = fig.add_gridspec(
        2, 1, left=0.072, right=0.985, top=0.88, bottom=0.19,
        height_ratios=[1.05, 0.92], hspace=0.56,
    )
    ax_top = fig.add_subplot(gs[0, 0])
    ax_bottom = fig.add_subplot(gs[1, 0], sharex=ax_top)

    x_ticks, x_labels = [], []
    for tissue_index, tissue in enumerate(DISPLAY_TISSUE_ORDER_G):
        one = table.loc[table["tissue"].eq(tissue)].sort_values(
            "rank_within_receiver_tissue"
        )
        start, end = blocks[tissue]
        for ax in (ax_top, ax_bottom):
            ax.axvspan(start - 0.48, end + 0.48,
                       color=block_colors[tissue_index % 2], zorder=-3)

        xs = [positions[(tissue, ct)] for ct in one["fine_cell_type"]]
        color = PAIR_COLOR_G[str(one["receptor_complex"].iloc[0])]
        ax_top.bar(
            xs, one["relative_expression_within_tissue"], width=0.56,
            color=color, alpha=0.80, edgecolor="none", zorder=2,
        )

        for x, delta in zip(xs, one["normalized_age_difference"]):
            delta = float(delta)
            delta_color = up if delta > 0 else down if delta < 0 else zero
            ax_bottom.plot([x, x], [0, delta], color=delta_color,
                           linewidth=1.55, zorder=2)
            ax_bottom.scatter(x, delta, s=18, color=delta_color,
                              edgecolor="white", linewidth=0.45, zorder=3)

        center = (start + end) / 2
        receptor = str(one["receptor_complex"].iloc[0])
        ax_top.text(center, 1.16, TISSUE_LABEL_G[tissue],
                    transform=ax_top.get_xaxis_transform(), ha="center",
                    fontsize=9.0, color=ink)
        ax_top.text(center, 1.055, f"VTN : {receptor}",
                    transform=ax_top.get_xaxis_transform(), ha="center",
                    fontsize=7.0, color=color)

        x_ticks.extend(xs)
        x_labels.extend([_short_panel_g_cell_type(x) for x in one["fine_cell_type"]])

    for ax in (ax_top, ax_bottom):
        ax.grid(axis="y", color=grid, linewidth=0.65, zorder=0)
        ax.grid(axis="x", visible=False)
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color("#53616D")
        ax.spines[["left", "bottom"]].set_linewidth(0.78)

    ax_top.set_ylim(0, 1.08)
    ax_top.set_yticks([0, 0.5, 1.0])
    ax_top.set_ylabel("Relative receptor-complex expression", fontsize=7.8)
    ax_top.set_xticks(x_ticks)
    ax_top.set_xticklabels(x_labels, rotation=38, ha="right", fontsize=6.6)
    ax_top.tick_params(axis="x", length=0, pad=5)

    ax_bottom.axhline(0, color="#7F8A94", linewidth=0.82, zorder=1)
    ax_bottom.set_ylim(-1.08, 1.08)
    ax_bottom.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax_bottom.set_yticklabels(["−1", "−0.5", "0", "+0.5", "+1"])
    ax_bottom.set_ylabel("Normalized age-group difference\n(>60 − <60)", fontsize=7.8)
    ax_bottom.set_xticks(x_ticks)
    ax_bottom.set_xticklabels(x_labels, rotation=38, ha="right", fontsize=6.6)
    ax_bottom.tick_params(axis="x", length=0, pad=5)
    ax_bottom.set_xlim(-0.75, cursor - 1.45)

    ax_bottom.legend(
        handles=[
            Line2D([0], [0], color=down, marker="o", markersize=4,
                   linewidth=1.5, label="Lower in >60"),
            Line2D([0], [0], color=up, marker="o", markersize=4,
                   linewidth=1.5, label="Higher in >60"),
        ],
        loc="upper right", bbox_to_anchor=(1.0, 1.25), frameon=False,
        ncol=2, fontsize=7, handlelength=1.4, columnspacing=1,
    )

    fig.text(0.012, 0.96, "g", fontsize=12, fontweight="bold", color=ink)
    PANEL_G_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    table.to_csv(
        PANEL_G_OUTPUT_STEM.with_name(PANEL_G_OUTPUT_STEM.name + "_data.csv"),
        index=False, encoding="utf-8-sig",
    )
    for suffix in ("png", "pdf", "svg"):
        fig.savefig(
            PANEL_G_OUTPUT_STEM.with_suffix(f".{suffix}"),
            dpi=600 if suffix == "png" else None,
            bbox_inches="tight", facecolor="white",
        )
    return fig


panel_g_table, panel_g_donor_means = prepare_panel_g_data(top_n_per_tissue=3)
panel_g_figure = plot_panel_g_vtn_receiver_remodeling(panel_g_table)
display(panel_g_figure)
plt.close(panel_g_figure)

print(f"Displayed cell types: {len(panel_g_table)}")
print(f"Saved: {PANEL_G_OUTPUT_STEM}.png/.pdf/.svg")
display(panel_g_table[[
    "tissue", "receptor_complex", "fine_cell_type",
    "relative_expression_within_tissue", "Age <60", "Age >60",
    "normalized_age_difference",
]])
